In [25]:
import cirq
import numpy as np
from typing import Sequence

In [26]:
BLOCK_A: Sequence[cirq.Qid] = cirq.LineQubit.range(9)
BLOCK_B: Sequence[cirq.Qid] = cirq.LineQubit.range(9, 18)

In [28]:
def encode_rotated_surface_block(qs: Sequence[cirq.Qid]) -> cirq.Circuit:
    """Encoding circuit for one 9-qubit rotated-surface block (matches the original)."""
    c = cirq.Circuit(
        [cirq.H.on_each(qs[i] for i in (1, 2, 6, 8))],
        cirq.CX(qs[1], qs[0]),
        cirq.CX(qs[6], qs[3]),
        cirq.CX(qs[4], qs[5]),
        cirq.CX(qs[8], qs[7]),
        cirq.CX(qs[1], qs[4]),
        cirq.CX(qs[5], qs[3]),
        cirq.CX(qs[1], qs[3]),
        cirq.CX(qs[2], qs[5]),
        cirq.CX(qs[8], qs[4]),
        cirq.CX(qs[8], qs[5]),
    )
    return c

In [29]:
def logical_cz(input_state: cirq.OP_TREE) -> cirq.Circuit:
    """
    Transversal CZ (via H–CNOT–H on the target block) for the rotated surface code
    using the encoding from Fig. 8 of https://arxiv.org/abs/2506.04084.
    Qubit 4 of each 9-qubit block carries the logical input."""
    c = cirq.Circuit(
        encode_rotated_surface_block(BLOCK_A),
        input_state,
        encode_rotated_surface_block(BLOCK_B),
        # Implement logical CZ by H–CNOT–H on the second block (transversal)
        cirq.H.on_each(BLOCK_B),
        [cirq.CX(a, b) for a, b in zip(BLOCK_A, BLOCK_B)],
        cirq.H.on_each(BLOCK_B),
        cirq.measure(*BLOCK_A, *BLOCK_B, key="m"),
    )
    return c

In [68]:
def logical_simulator(
    physical_circuit: cirq.Circuit, repetitions: int
) -> cirq.ResultDict:
    sim = cirq.Simulator()
    logical_results = []
    results = sim.run(physical_circuit, repetitions=repetitions)
    for result in results.measurements["m"]:
        logical_0_result = sum([result[1], result[4], result[7]]) % 2
        logical_1_result = sum([result[10], result[13], result[16]]) % 2
        logical_results.append([logical_0_result, logical_1_result])

    logical_result_dict = cirq.ResultDict(
        params=cirq.ParamResolver({}), measurements={"m": np.array(logical_results)}
    )
    return logical_result_dict

In a CZ gate the target should always be $|0\rangle$ no matter the input on the control. This test inputs $|0\rangle$, $|1\rangle$, $|+\rangle$, $|-\rangle$, $|i\rangle$, $|-i\rangle$ and confirms that the value of the target is $|0\rangle$ when measured.

In [81]:
def test_logical_cz() -> None:
    repetitions = 10
    # input is |0⟩
    results = logical_simulator(logical_cz(cirq.I(BLOCK_A[4])), repetitions)
    assert all(np.all(vals == 0) for vals in results.measurements.values())

    # Input is |1⟩
    results = logical_simulator(logical_cz(cirq.X(BLOCK_A[4])), repetitions)
    vals = results.measurements["m"]
    assert np.all(
        vals[:, 1] == 0
    ), f"Second qubit under key 'm' is not all zeroes: {vals[:, 1]}"

    # Input is |+⟩
    results = logical_simulator(logical_cz(cirq.H(BLOCK_A[4])), repetitions)
    vals = results.measurements["m"]
    assert np.all(
        vals[:, 1] == 0
    ), f"Second qubit under key 'm' is not all zeroes: {vals[:, 1]}"

    # Input is |-⟩
    results = logical_simulator(
        logical_cz([cirq.H(BLOCK_A[4]), cirq.Z(BLOCK_A[4])]), repetitions
    )
    vals = results.measurements["m"]
    assert np.all(
        vals[:, 1] == 0
    ), f"Second qubit under key 'm' is not all zeroes: {vals[:, 1]}"

    # Input is |i⟩
    results = logical_simulator(
        logical_cz([cirq.H(BLOCK_A[4]), cirq.S(BLOCK_A[4])]), repetitions
    )
    vals = results.measurements["m"]
    assert np.all(
        vals[:, 1] == 0
    ), f"Second qubit under key 'm' is not all zeroes: {vals[:, 1]}"

    # Input is |i⟩
    results = logical_simulator(
        logical_cz([cirq.H(BLOCK_A[4]), cirq.S(BLOCK_A[4]) ** -1]), repetitions
    )
    vals = results.measurements["m"]
    assert np.all(
        vals[:, 1] == 0
    ), f"Second qubit under key 'm' is not all zeroes: {vals[:, 1]}"

In [82]:
test_logical_cz()

In [83]:
def logical_cx(input_state: cirq.OP_TREE) -> cirq.Circuit:
    """
    Transversal CZ (via H–CNOT–H on the target block) for the rotated surface code
    using the encoding from Fig. 8 of https://arxiv.org/abs/2506.04084.
    Qubit 4 of each 9-qubit block carries the logical input."""
    c = cirq.Circuit(
        encode_rotated_surface_block(BLOCK_A),
        input_state,
        encode_rotated_surface_block(BLOCK_B),
        [cirq.CX(a, b) for a, b in zip(BLOCK_A, BLOCK_B)],
        cirq.measure(*BLOCK_A, *BLOCK_B, key="m"),
    )
    return c

In [104]:
def test_logical_cx() -> None:
    repetitions = 10
    # input is |0⟩
    results = logical_simulator(logical_cx(cirq.I(BLOCK_A[4])), repetitions)
    assert all(np.all(vals == 0) for vals in results.measurements.values())

    # Input is |1⟩
    results = logical_simulator(logical_cx(cirq.X(BLOCK_A[4])), repetitions)
    vals = results.measurements["m"]

    # Input is |+⟩
    results = logical_simulator(logical_cx(cirq.H(BLOCK_A[4])), repetitions)
    vals = results.measurements["m"]
    assert np.array_equal(
        vals[:, 0], vals[:, 1]
    ), f"First and second qubit differ:\nq0={vals[:, 0]}\nq1={vals[:, 1]}"

    # Input is |-⟩
    results = logical_simulator(
        logical_cx([cirq.H(BLOCK_A[4]), cirq.Z(BLOCK_A[4])]), repetitions
    )
    vals = results.measurements["m"]
    assert np.array_equal(
        vals[:, 0], vals[:, 1]
    ), f"First and second qubit differ:\nq0={vals[:, 0]}\nq1={vals[:, 1]}"

    # Input is |i⟩
    results = logical_simulator(
        logical_cx([cirq.H(BLOCK_A[4]), cirq.S(BLOCK_A[4])]), repetitions
    )
    vals = results.measurements["m"]
    assert np.array_equal(
        vals[:, 0], vals[:, 1]
    ), f"First and second qubit differ:\nq0={vals[:, 0]}\nq1={vals[:, 1]}"

    # Input is |i⟩
    results = logical_simulator(
        logical_cx([cirq.H(BLOCK_A[4]), cirq.S(BLOCK_A[4]) ** -1]), repetitions
    )
    vals = results.measurements["m"]
    assert np.array_equal(
        vals[:, 0], vals[:, 1]
    ), f"First and second qubit differ:\nq0={vals[:, 0]}\nq1={vals[:, 1]}"

In [105]:
test_logical_cx()